# XMOFS-Deep: Ovarian cancer (TCGA-OV) on Colab
Run the cells **one by one** (Shift+Enter), top to bottom. Do NOT use "Run all".

**Before cell 1:** Runtime -> Change runtime type -> **T4 GPU** -> Save, then click **Connect**.
Upload `xmofs_deep_code.zip` (the new one) to the Files panel (left folder icon).

## Cell 1: setup

In [ ]:
import os, subprocess, sys
subprocess.run("nvidia-smi -L", shell=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymoo", "shap", "gseapy"], check=True)
subprocess.run("unzip -o -q xmofs_deep_code.zip", shell=True, check=True)
n = len([f for f in os.listdir(".") if f.endswith(".py")])
print(n, "scripts ready (expect 13)")

def save(tag):
    # zip the results (without big model files) and download them to your computer
    subprocess.run(f'zip -r -q results_ov_{tag}.zip results_ov cleaned_data_ov/ov_label_quality.csv -x "*.pt"', shell=True)
    from google.colab import files
    files.download(f"results_ov_{tag}.zip")

## Cell 2: download the data (about 1-3 minutes)

In [ ]:
!wget -q -O TCGA_489_UE.k4.txt https://api.gdc.cancer.gov/data/c7a3c958-c576-48a7-bbd0-543fbf6ea218
!wget -q -O TCGA_489_UE.k4.posSilhouette.txt https://api.gdc.cancer.gov/data/9d33ce5c-2a34-4bef-ae09-419a53f2feed
!wget -q https://datahub.assets.cbioportal.org/ov_tcga_pan_can_atlas_2018.tar.gz
!tar -xzf ov_tcga_pan_can_atlas_2018.tar.gz
!ls -la TCGA_489_UE.k4.txt ov_tcga_pan_can_atlas_2018/data_mrna_seq_v2_rsem.txt
!head -3 TCGA_489_UE.k4.txt

## Cell 3: clean the data. **Send me this output** (expect: in both = 291; classes 79 / 75 / 71 / 66)

In [ ]:
!python prep_ov.py

## Optional cell: restore after a disconnect
Only if Colab disconnected: re-run cells 1-3, upload the last `results_ov_*.zip` you downloaded, then run this cell.
Finished stages are skipped automatically.

In [ ]:
import glob, subprocess
z = sorted(glob.glob("results_ov_*.zip"))
print("found:", z)
if z:
    subprocess.run(["unzip", "-o", "-q", z[-1]])
    print(sorted(__import__("os").listdir("results_ov")))
else:
    print("No results zip uploaded yet. Upload the last results_ov_*.zip you downloaded, then run this cell again.")

## Cell 4 (first time only): quick smoke test, about 10-20 minutes
Only checks that every stage runs. **Never report these numbers.** Skip this cell if you already did it once.

In [ ]:
!python xmofs_runner.py --dataset ov --profile quick

## Cells 5-8: the real (full) run
Keep this tab open. Every cell downloads a zip when it finishes: keep those zips.
If a cell stops because of a disconnect, use the restore cell and run the same cell again.

### Cell 5: baselines (stages 3 and 4)

In [ ]:
!python xmofs_runner.py --dataset ov --profile full --stages 3,4
save("stages3-4")

### Cell 6: NSGA-II search (stage 6): the longest stage

In [ ]:
!python xmofs_runner.py --dataset ov --profile full --stages 6
save("stage6")

### Cell 7: hypervolume comparison (stage 7)

In [ ]:
!python xmofs_runner.py --dataset ov --profile full --stages 7
save("stage7")

### Cell 8: SHAP and repeated-split statistics (stages 8 and 9)

In [ ]:
!python xmofs_runner.py --dataset ov --profile full --stages 8,9
!cat results_ov/chosen_point.json
save("stages8-9")

## Cell 9: results summary. **Send me this output**

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results_ov/FINAL_SUMMARY.md").read()))

## Optional cell: pathway enrichment again (only if Stage 8 printed a 429 error)

In [ ]:
!sed -i 's#stage8_results/#results_ov/stage8_results/#g' stage8_enrichment_only.py
!python stage8_enrichment_only.py